# Production Planning with Perishable Inventory

**Problem.** A product has known demand over the next 12 months. Each month the plant can make up to 800 units on regular time and a further 200 units on overtime. Regular units cost `$10` each, overtime units cost `$12`. Anything not sold in the month it is made goes into storage at `$1` per unit per month, but the product is perishable, so a unit can be held for <u>at most two months</u> beyond the month it was produced.

**Question:** how much should be produced each month, in each mode, and when should it be sold, to minimise total production and storage cost over the year?

The interesting feature is the two-month shelf life. It means the plan cannot simply build a large buffer early and draw it down, the inventory has an expiry date, so pre-building has to be timed to land within two months of the demand it serves.

## Formulation

**Sets.** &nbsp; Production month $i \in \{1,\ldots,12\}$, &nbsp; Production mode $j \in \{\text{regular},\ \text{overtime}\}$, &nbsp; Sale month $k \in \{i,\ i+1,\ i+2\}$ with $k \le 12$.

**Parameters.**

| Symbol | Meaning |
| --- | --- |
| $d_k$ | Demand in month $k$ |
| $u_j$ | Monthly capacity of mode $j$ |
| $c_j$ | Production cost per unit in mode $j$ |
| $h$ | Storage cost per unit per month |

**Decision variable.** &ensp;  $x_{ijk} \ge 0$ = units produced in month $i$, using mode $j$, and sold in month $k$.

**Model.**

$$
\begin{aligned}
\min \quad & \sum_{i}\sum_{j}\sum_{k} \big(c_j + h\,(k-i)\big)\, x_{ijk} \\
\text{s.t.}\quad
& \sum_{i}\sum_{j} x_{ijk} \;\ge\; d_k && \forall \; k && \text{(demand is met)}\\
& \sum_{k} x_{ijk} \;\le\; u_j && \forall \; i, j && \text{(monthly capacity of each mode)}\\
& x_{ijk} \ge 0, \qquad \quad k \in \{i,\ i+1,\ i+2\},\ \ k \le 12.
\end{aligned}
$$

A unit cannot be sold before it is made ($k \ge i$), and cannot be held more than two months ($k \le i+2$). Also, for the storage cost, units produced in month $i$ and sold in month $k$ is stored for $\left(k-i\right)$ months

In [1]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## Data

In [2]:
MONTHS = list(range(1, 13))
MODES = ["regular", "overtime"]
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

demand = pd.Series([800, 900, 1100, 400, 700, 1100, 700, 900, 800, 700, 800, 1100], index=MONTHS)
capacity = dict(zip(MODES, [800, 200]))     # units per month
cost = dict(zip(MODES, [10, 12]))           # $ per unit
HOLDING = 1        # $ per unit per month
MAX_HOLD = 2       # months a unit may be held beyond production

pd.DataFrame({"month": MONTH_NAMES, "demand": demand.values}, index=MONTHS).T

,1,2,3,4,5,6,7,8,9,10,11,12
month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
demand,800,900,1100,400,700,1100,700,900,800,700,800,1100


## Model

In [3]:
def sale_months(i):
    '''Months a unit produced in month i can be sold in.'''
    return [k for k in MONTHS if i <= k <= i + MAX_HOLD]

def production_months(k):
    '''Months a unit sold in month k could have been produced in.'''
    return [i for i in MONTHS if k - MAX_HOLD <= i <= k]

In [4]:
env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

m = gp.Model("Production_planning", env=env)

x = m.addVars([(i, j, k) for i in MONTHS for j in MODES for k in sale_months(i)], lb=0.0, name="x")

m.setObjective(gp.quicksum((cost[j] + HOLDING * (k - i)) * x[i, j, k]
                           for i in MONTHS for j in MODES for k in sale_months(i)), GRB.MINIMIZE)

m.addConstrs((gp.quicksum(x[i, j, k] for i in production_months(k) for j in MODES) >= demand[k]
              for k in MONTHS), name="demand")

m.addConstrs((gp.quicksum(x[i, j, k] for k in sale_months(i)) <= capacity[j]
              for i in MONTHS for j in MODES), name="capacity")

m.optimize()
print(f"Optimal total production and storage cost: ${m.ObjVal:,.2f}")
print(f"Model size         : {len(x)} variables " f"(a full i x j x k grid would be {12 * 2 * 12})")

Optimal total production and storage cost: $102,100.00
Model size         : 66 variables (a full i x j x k grid would be 288)


Because variables are created only for valid  $(i, j, k)$ combinations, no forced-to-zero variables are needed to enforce the shelf life.

## Solution

In [5]:
plan = pd.DataFrame(
    [{"produced in": MONTH_NAMES[i - 1], "type": j, "sold in": MONTH_NAMES[k - 1],
      "units": x[i, j, k].X}
     for i in MONTHS for j in MODES for k in sale_months(i) if x[i, j, k].X > 1e-6])
plan

,produced in,type,sold in,units
0,Jan,regular,Jan,800.0
1,Feb,regular,Feb,800.0
2,Feb,overtime,Feb,100.0
3,Feb,overtime,Mar,100.0
4,Mar,regular,Mar,800.0
5,Mar,overtime,Mar,200.0
6,Apr,regular,Apr,400.0
7,Apr,regular,May,200.0
8,May,regular,May,500.0
9,May,regular,Jun,300.0


In [7]:
schedule = pd.DataFrame({
    "month": MONTH_NAMES,
    "demand": demand.values,
    "regular": [sum(x[i, "regular", k].X for k in sale_months(i)) for i in MONTHS],
    "overtime": [sum(x[i, "overtime", k].X for k in sale_months(i)) for i in MONTHS],
}, index=MONTHS)
schedule["produced"] = schedule.regular + schedule.overtime
schedule["ending inventory"] = [
    sum(x[i, j, k].X for i in MONTHS for j in MODES for k in sale_months(i) if i <= t < k)
    for t in MONTHS
]
schedule

,month,demand,regular,overtime,produced,ending inventory
1,Jan,800,800.0,0.0,800.0,0.0
2,Feb,900,800.0,200.0,1000.0,100.0
3,Mar,1100,800.0,200.0,1000.0,0.0
4,Apr,400,600.0,0.0,600.0,200.0
5,May,700,800.0,0.0,800.0,300.0
6,Jun,1100,800.0,0.0,800.0,0.0
7,Jul,700,800.0,0.0,800.0,100.0
8,Aug,900,800.0,0.0,800.0,0.0
9,Sep,800,800.0,0.0,800.0,0.0
10,Oct,700,800.0,0.0,800.0,100.0


## Conclusion

The optimal plan costs **`$102,100`** over the 12 months: 9,400 units on regular time, 600 on overtime, and 900 unit-months of storage.

**Regular time carries almost everything.** Only 200 of the 9,600 available regular-time units go unused (in April, the lowest-demand month), so the plant runs essentially flat out at `$10` per unit and treats overtime as a fallback. Just 6% of output is made on overtime.

**Storage is what makes the plan feasible, not just cheaper.** March, June and December each demand 1,100 units, but a single month can produce at most 800 + 200 = 1,000 even on full overtime. Those months cannot be served from their own production, so some pre-building
is mandatory, without storage the problem has no solution at all.

**Holding is cheaper than overtime, and the plan exploits that.** Carrying a unit one month costs `$1`, while producing on overtime costs `$2` more per unit. So wherever an earlier month has spare regular capacity, the model builds early rather than paying the overtime premium: 900 units are carried forward, 100 from February, 200 from April, 300 from May, and 100 each from July, October and November.

**Overtime is used in only three months, i.e. February, March and December.** These are the months where pre-building is not an option: the preceding months have no spare regular capacity left, having already consumed it on their own demand. December is the clearest case, since it is the last month in the horizon and cannot borrow from anything later.

**No unit is ever held more than one month**, even though two are allowed, 9,100 units are sold in the month they are produced and 900 are held for exactly one month. The perishability limit therefore does not bind at these demand levels; the binding restrictions are monthly capacity and the `$2` overtime premium.